# 混合精度与性能边界

## 学习目标

能够解释 autocast 和梯度缩放的职责，并在无 CUDA 时安全跳过。


## 概念模型与执行路径

autocast 根据算子选择较低或较高精度；GradScaler 放大损失，降低 float16 小梯度下溢风险。AMP 主要面向 CUDA，不能假设所有设备具有相同行为。


### 实验 1：探测可用加速设备

**实验目的**：在运行 CUDA 专属 AMP 示例前明确当前硬件能力。`torch.cuda.is_available()` 表示 PyTorch 能否使用 CUDA；`torch.backends.mps.is_available()` 表示 Apple Metal 后端是否可用。

设备可用性取决于硬件、驱动和 PyTorch 构建。CUDA 为 False 不是 notebook 失败，而是后续 CUDA autocast/GradScaler 实验应安全跳过。MPS 可用也不等于能够原样执行 CUDA float16 AMP 路径，不同后端的算子覆盖和数值策略不同。

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())


### 实验 2：观察 CUDA autocast 的算子精度选择

**实验目的**：在 CUDA 可用时创建 float32 线性层和输入，并在 `torch.autocast(device_type='cuda',dtype=float16)` 作用域内前向，观察输出通常变为 float16。无 CUDA 时打印跳过原因。

autocast 不会永久把模型参数转换成 half；它按算子策略选择输入/输出计算精度。矩阵乘法等适合低精度的算子可用 float16/Tensor Core，而数值敏感算子可能保留更高精度。离开上下文后，默认精度行为恢复。

**边界**：不要假设上下文内所有张量都是 float16，也不要仅凭一个输出 dtype 推断整个模型的计算精度。

In [ ]:
if torch.cuda.is_available():
    model = torch.nn.Linear(16, 4).cuda()
    inputs = torch.randn(8, 16, device="cuda")
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(inputs)
    print("autocast output dtype:", outputs.dtype)
else:
    print("跳过 CUDA autocast；其他课程仍可在 CPU/MPS 运行。")


### 实验 3：使用 GradScaler 完成一次 AMP 参数更新

**实验目的**：展示 CUDA float16 训练步骤的正确顺序：清梯度 → autocast 前向与 loss → scale(loss).backward → scaler.step → scaler.update。

float16 可表示的非零范围有限，小梯度可能下溢为 0。GradScaler 先把 loss 乘动态 scale，使反向梯度整体放大；`scaler.step` 在更新前反缩放并检查 inf/NaN，若发现溢出会跳过 optimizer step；`update` 再调整下一轮 scale。模型主参数与 optimizer state 通常保持 float32。

targets 由实验 2 的 outputs shape 创建，但 loss 中重新执行一次模型前向，因此实验 2 必须先成功运行。CUDA 不可用时整个分支跳过，不会产生 model、outputs、optimizer 等变量，这正是安全降级设计。

**扩展**：使用梯度裁剪时，应先 `scaler.unscale_(optimizer)` 再 clip；保存恢复训练时也应保存 scaler state。

In [ ]:
if torch.cuda.is_available():
    optimizer = torch.optim.AdamW(model.parameters())
    scaler = torch.amp.GradScaler("cuda")
    targets = torch.randn_like(outputs)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss = torch.nn.functional.mse_loss(model(inputs), targets)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    print("scaled step complete:", loss.item())


### 实验 4：运行设备感知的 AMP 命令行示例

**实验目的**：运行一轮合成图像分类 AMP 更新，或在非 CUDA 设备上明确跳过。`--device auto` 可能选择 CUDA、MPS 或 CPU，但脚本只有当最终 device 类型是 CUDA 时才进入 AMP 训练。

CUDA 路径创建 `Flatten → Linear(3072,10)`、AdamW、GradScaler 和最多 16 张 `3×32×32` 图像，在 autocast 下计算交叉熵并完成 scaled step。非 CUDA 路径打印说明后正常退出，因此适合跨机器冒烟。`--quick` 在该脚本中不会改变核心逻辑；实际 batch 由 `min(config.batch_size,16)` 限制。

**性能边界**：这个模型很小，AMP 可能因 cast 和 scaler 管理开销反而更慢。性能结论必须用足够规模的代表性模型、预热、重复测量和 CUDA 同步得出。

In [ ]:
# python 07-deep-learning/pytorch/examples/mixed_precision.py --device auto --quick


## 底层机制

混合精度把三个问题分开：autocast 选择算子精度以提高吞吐、降低激活显存；GradScaler 保护 float16 反向中的小梯度；optimizer 仍更新通常为 float32 的主参数。bfloat16 指数范围接近 float32，通常不需要与 float16 相同的梯度缩放，但有效精度和硬件支持不同。

梯度缩放只缓解数值表示下溢，不会修复错误学习率、爆炸激活、错误 loss 或不稳定模型。性能收益取决于 Tensor Core 支持、矩阵尺寸、batch size、内存带宽和数据管线；测量峰值显存前需重置统计，计时前后要同步 CUDA。

## 检查点

回答并验证：1）autocast 与 GradScaler 分别解决什么问题？2）参数会被永久转为 float16 吗？3）`scaler.step` 检测到 inf 时会怎样？4）梯度裁剪为何要先 unscale？5）CUDA 不可用时跳过为何不是失败？6）MPS 可用为何不等同于 CUDA AMP 可用？7）小模型为什么可能没有加速？

## 试一试

在 CUDA 上比较 FP32 与 AMP 的迭代时间、吞吐和峰值显存：固定输入和模型，先预热，再多次运行，计时边界调用 `torch.cuda.synchronize()`，显存统计前调用 reset API。记录 scaler 的 scale 变化并故意构造溢出，观察 step 是否跳过；再加入梯度裁剪，验证正确的 unscale 顺序。

## 常见错误与调试

- **手动把全部参数转成 half**：敏感算子和更新可能不稳定；让 autocast 管理算子精度。
- **在 CPU/MPS 强行使用 CUDA scaler**：后端契约错误；按 device 分支。
- **忘记 scale/backward/step/update 顺序**：缩放失效或更新错误；遵循标准模板。
- **缩放梯度后直接裁剪**：阈值作用于放大梯度；先 unscale。
- **只保存 optimizer 不保存 scaler**：恢复后动态 scale 丢失；一并 checkpoint。
- **CUDA 异步计时未同步**：测到提交时间而非完成时间；边界同步。
- **没有预热或只测一次**：初始化噪声主导结果；预热并重复统计。
- **认为 AMP 必然更快**：小模型/cast 开销可能占主导；以实测为准。